# Ch.4 — SHAP Interpretability

## EnsembleAI Challenge — Ch.4

**Constraint #4 (INTERPRETABILITY)**: Explain every ensemble prediction.

SHAP (SHapley Additive exPlanations) decomposes any prediction into per-feature contributions:

$$f(\mathbf{x}) = E[f] + \sum_i \phi_i$$

## Setup & Data

In [ ]:
# ── Setup & Data ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error

IMG = "img/"
import os; os.makedirs(IMG, exist_ok=True)

np.random.seed(42)

data = fetch_california_housing()
X, y = data.data, data.target
feature_names = list(data.feature_names)
y_cls = (y > np.median(y)).astype(int)

X_tr, X_te, y_tr, y_te, y_tr_cls, y_te_cls = train_test_split(
    X, y, y_cls, test_size=0.2, random_state=42)

print(f"Train: {len(X_tr):,}  Test: {len(X_te):,}")
shap.initjs()

## Train XGBoost (Regression + Classification)

In [ ]:
# ── Train Models ─────────────────────────────────────────────────────────────
xgb_reg = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                         random_state=42, verbosity=0)
xgb_reg.fit(X_tr, y_tr)

xgb_cls = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=4,
                          random_state=42, verbosity=0, eval_metric='logloss')
xgb_cls.fit(X_tr, y_tr_cls)

rmse = np.sqrt(mean_squared_error(y_te, xgb_reg.predict(X_te)))
print(f"XGBoost Regression RMSE: {rmse:.4f}")
print(f"XGBoost Classification Accuracy: {xgb_cls.score(X_te, y_te_cls):.4f}")

## SHAP — Regression: Waterfall Plot

In [ ]:
# ── SHAP Waterfall (Regression) ──────────────────────────────────────────────
explainer_reg = shap.TreeExplainer(xgb_reg)
shap_values_reg = explainer_reg(X_te, feature_names=feature_names)

idx = 0
pred = xgb_reg.predict(X_te[idx:idx+1])[0]
base = shap_values_reg[idx].base_values
print(f"Sample {idx}: prediction={pred:.2f}, base={base:.2f}")
print(f"SHAP sum check: base + sum(shap) = {base + shap_values_reg[idx].values.sum():.2f}")

fig = shap.plots.waterfall(shap_values_reg[idx], show=False)
plt.tight_layout()
plt.savefig(f'{IMG}ch04_waterfall_regression.png', dpi=150, bbox_inches='tight')
plt.show()

## SHAP — Beeswarm (Global Feature Importance)

In [ ]:
# ── SHAP Beeswarm (Global) ───────────────────────────────────────────────────
fig = shap.plots.beeswarm(shap_values_reg, show=False)
plt.tight_layout()
plt.savefig(f'{IMG}ch04_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print("Beeswarm: each dot = one sample. Color = feature value. X = SHAP value.")

## SHAP — Bar Plot (Mean Absolute SHAP)

In [ ]:
# ── SHAP Bar Plot ────────────────────────────────────────────────────────────
fig = shap.plots.bar(shap_values_reg, show=False)
plt.tight_layout()
plt.savefig(f'{IMG}ch04_bar_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## SHAP — Dependence Plot (Non-linear Effects)

In [ ]:
# ── SHAP Dependence Plot ─────────────────────────────────────────────────────
fig = shap.plots.scatter(shap_values_reg[:, "MedInc"], show=False)
plt.tight_layout()
plt.savefig(f'{IMG}ch04_dependence_medinc.png', dpi=150, bbox_inches='tight')
plt.show()

print("Non-linear: MedInc > 5 has steep positive SHAP; below 3 is flat.")

## SHAP — Classification Waterfall

In [ ]:
# ── SHAP for Classification ──────────────────────────────────────────────────
explainer_cls = shap.TreeExplainer(xgb_cls)
shap_values_cls = explainer_cls(X_te, feature_names=feature_names)

print(f"Classification sample {idx}: actual={y_te_cls[idx]}, pred={xgb_cls.predict(X_te[idx:idx+1])[0]}")
print(f"Base (log-odds): {shap_values_cls[idx].base_values:.3f}")

fig = shap.plots.waterfall(shap_values_cls[idx], show=False)
plt.tight_layout()
plt.savefig(f'{IMG}ch04_waterfall_classification.png', dpi=150, bbox_inches='tight')
plt.show()

## SHAP Additivity Check

Key property: $f(\mathbf{x}) = E[f] + \sum_i \phi_i$ (efficiency axiom).

In [ ]:
# ── Additivity Check ─────────────────────────────────────────────────────────
for i in range(5):
    pred = xgb_reg.predict(X_te[i:i+1])[0]
    shap_sum = shap_values_reg[i].base_values + shap_values_reg[i].values.sum()
    print(f"Sample {i}: prediction={pred:.4f}  base+SHAP={shap_sum:.4f}  diff={abs(pred-shap_sum):.6f}")

print("\nSHAP values sum exactly to prediction - base value (efficiency axiom). ✅")

## Progress Check

| # | Constraint | Status | Evidence |
|---|-----------|--------|----------|
| 1 | IMPROVEMENT | ✅ | (Ch.1-3) |
| 2 | DIVERSITY | ✅ | (Ch.1-3) |
| 3 | EFFICIENCY | ✅ | TreeSHAP is milliseconds |
| 4 | INTERPRETABILITY | ✅ | **Per-prediction SHAP explanations** |
| 5 | ROBUSTNESS | ✅ | (Ch.1-3) |

**All 5 constraints addressed!**

**Next**: Ch.5 — Stacking combines diverse models via a meta-learner.

## Exercises

1. **SHAP for Random Forest**: Create a `shap.TreeExplainer` for a Random Forest. Compare its beeswarm to XGBoost's. Do they rank features the same?
2. **Top-3 influential features per sample**: For the first 100 test samples, find the top-3 features by |SHAP|. How often is MedInc in the top 3?
3. **SHAP interaction values**: Use `shap.TreeExplainer(model).shap_interaction_values(X)` to find the strongest feature interaction pair.

In [ ]:
# ── Exercise 1: SHAP for Random Forest ───────────────────────────────────────
# TODO: from sklearn.ensemble import RandomForestRegressor
#     rf = RandomForestRegressor(n_estimators=200, random_state=42)
#     rf.fit(X_tr, y_tr)
#     explainer_rf = shap.TreeExplainer(rf)
#     shap_rf = explainer_rf(X_te)
#     shap.plots.beeswarm(shap_rf)
pass

In [ ]:
# ── Exercise 2: Top-3 features per sample ───────────────────────────────────
# TODO: for i in range(100):
#     top3 = np.argsort(np.abs(shap_values_reg[i].values))[-3:]
#     check if MedInc index is in top3
pass

In [ ]:
# ── Exercise 3: SHAP interaction values ─────────────────────────────────────
# TODO: interaction_values = explainer_reg.shap_interaction_values(X_te[:100])
#     Find the (i,j) pair with largest mean |interaction|
pass